In [1]:
import re
import pymorphy3
import pandas as pd
from sklearn.model_selection import train_test_split
from razdel import tokenize
from navec import Navec
import numpy as np
from math import log
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
seed = 42
np.random.seed(seed)

# 1. Загружаем датасет
Возьмём его часть (половину)

In [3]:
ds = pd.read_csv(r'../../data/petitions.csv')

In [4]:
ds.head()

,id,public_petition_text,reason_category
0,3168490,снег на дороге,Благоустройство
1,3219678,очистить кабельный киоск от рекламы,Благоустройство
2,2963920,"Просим убрать все деревья и кустарники, которы...",Благоустройство
3,3374910,Неудовлетворительное состояние парадной - надп...,Содержание МКД
4,3336285,Граффити,Благоустройство


In [5]:
ds = ds.drop(columns=['id'], axis=1)

In [6]:
ds['reason_category'].value_counts()

reason_category
Благоустройство                                                                     34769
Содержание МКД                                                                      14461
Нарушение правил пользования общим имуществом                                        2170
Незаконная информационная и (или) рекламная конструкция                              1831
Фасад                                                                                1483
Повреждения или неисправность элементов уличной инфраструктуры                       1164
Кровля                                                                                825
Водоснабжение                                                                         809
Состояние рекламных или информационных конструкций                                    673
Санитарное состояние                                                                  434
Центральное отопление                                                               

In [7]:
top_classes = ds['reason_category'].value_counts().head(2).index

frames = []
for cls, group in ds.groupby('reason_category'):
    if cls in top_classes:
        group = group.sample(frac=0.3, random_state=42)  # оставить половину
    frames.append(group)

ds_sample = pd.concat(frames)

In [8]:
ds_sample['reason_category'].value_counts()

reason_category
Благоустройство                                                                     10431
Содержание МКД                                                                       4338
Нарушение правил пользования общим имуществом                                        2170
Незаконная информационная и (или) рекламная конструкция                              1831
Фасад                                                                                1483
Повреждения или неисправность элементов уличной инфраструктуры                       1164
Кровля                                                                                825
Водоснабжение                                                                         809
Состояние рекламных или информационных конструкций                                    673
Санитарное состояние                                                                  434
Центральное отопление                                                               

In [9]:
ds_sample.head()

,public_petition_text,reason_category
57338,упавшее дерево,Благоустройство
21420,Повреждена решетка стока.,Благоустройство
39101,"Неубранный снег на проезжей части и там, где ...",Благоустройство
7870,Прошу установить по лежачему полицейскому с ка...,Благоустройство
42523,объявления на знаках,Благоустройство


# 2. Выполним предобработку данных

In [10]:
ds_sample['public_petition_text'] = ds_sample['public_petition_text'].map(lambda x: re.sub(r'[^а-яё\s]', '', x, flags=re.IGNORECASE))

In [11]:
ds_sample['public_petition_text'] = ds_sample['public_petition_text'].map(lambda x: x.lower())

In [12]:
ds_sample['public_petition_text'] = ds_sample['public_petition_text'].map(lambda x: re.sub(r'\s+', ' ', x).strip())

In [13]:
ds_sample['public_petition_text'] = ds_sample['public_petition_text'].map(lambda x: '\n'.join([line.strip() for line in x.split('\n')]))

In [14]:
ds_sample.head()

,public_petition_text,reason_category
57338,упавшее дерево,Благоустройство
21420,повреждена решетка стока,Благоустройство
39101,неубранный снег на проезжей части и там где па...,Благоустройство
7870,прошу установить по лежачему полицейскому с ка...,Благоустройство
42523,объявления на знаках,Благоустройство


In [15]:
def get_tokens(string):
    tokens = list(tokenize(string))
    return [_.text for _ in tokens]

In [16]:
ds_sample['public_petition_text'] = ds_sample['public_petition_text'].map(lambda x: get_tokens(x))

In [17]:
ds_sample.head()

,public_petition_text,reason_category
57338,"[упавшее, дерево]",Благоустройство
21420,"[повреждена, решетка, стока]",Благоустройство
39101,"[неубранный, снег, на, проезжей, части, и, там...",Благоустройство
7870,"[прошу, установить, по, лежачему, полицейскому...",Благоустройство
42523,"[объявления, на, знаках]",Благоустройство


In [18]:
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Denis\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Denis\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [19]:
stop_words = set(stopwords.words('russian'))

In [20]:
def delete_stopwords(tokens):
    return [w for w in tokens if w.isalpha() and w not in stop_words]

In [21]:
ds_sample['public_petition_text'] = ds_sample['public_petition_text'].map(lambda x: delete_stopwords(x))

In [22]:
ds_sample.head()

,public_petition_text,reason_category
57338,"[упавшее, дерево]",Благоустройство
21420,"[повреждена, решетка, стока]",Благоустройство
39101,"[неубранный, снег, проезжей, части, паркуются,...",Благоустройство
7870,"[прошу, установить, лежачему, полицейскому, ка...",Благоустройство
42523,"[объявления, знаках]",Благоустройство


In [23]:
morph = pymorphy3.MorphAnalyzer()
def lemmatize_words(words: list) -> list:
    return [morph.parse(word)[0].normal_form for word in words]

In [24]:
ds_sample['public_petition_text'] = ds_sample['public_petition_text'].map(lambda x: lemmatize_words(x))

In [25]:
ds_sample.head()

,public_petition_text,reason_category
57338,"[упавший, дерево]",Благоустройство
21420,"[повредить, решётка, сток]",Благоустройство
39101,"[неубранный, снег, проезжий, часть, парковатьс...",Благоустройство
7870,"[просить, установить, лежачий, полицейский, ка...",Благоустройство
42523,"[объявление, знак]",Благоустройство


In [26]:
tfidf_vectorizer = TfidfVectorizer()
ds_sample['public_petition_text'] = ds_sample['public_petition_text'].map(lambda x: ' '.join(x))

In [27]:
ds_sample.head()

,public_petition_text,reason_category
57338,упавший дерево,Благоустройство
21420,повредить решётка сток,Благоустройство
39101,неубранный снег проезжий часть парковаться авт...,Благоустройство
7870,просить установить лежачий полицейский каждый ...,Благоустройство
42523,объявление знак,Благоустройство


In [28]:
X_tfidf = tfidf_vectorizer.fit_transform(ds_sample['public_petition_text'])

In [29]:
df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())
df_tfidf.head()

,аа,аао,ааш,аба,абз,абп,абразивный,абрамов,абсолютно,абсолютный,...,яхтенный,ячеистый,ячейка,ящик,ящикитребоваться,ящикпогнутый,ёж,ёлка,ёлочный,ёмкость
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [30]:
X = df_tfidf
y = ds_sample['reason_category']

In [31]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y)

# Обучим модель

In [32]:
from catboost import CatBoostClassifier
from sklearn.metrics import classification_report, accuracy_score

In [33]:
unique, counts = np.unique(y_train, return_counts=True)
total = len(y_train)
class_weights = [total / (len(unique) * count) for count in counts]

In [36]:
model = CatBoostClassifier(
    iterations=1000,             # больше итераций + ранняя остановка
    learning_rate=0.03,          # чуть меньше для стабильности
    depth=8,                     # немного глубже, чтобы уловить сложные паттерны
    l2_leaf_reg=5,               # регуляризация (можно 3–10 подбирать)
    border_count=128,            # повышает качество на сложных фичах
    loss_function='MultiClass',  
    class_weights=class_weights, # важно при дисбалансе!
    random_seed=42,
    early_stopping_rounds=50,    # чтобы не переобучаться
    eval_metric='TotalF1',       # лучше отражает баланс Precision/Recall
    task_type='CPU',             # 'GPU' если доступна видеокарта
    bootstrap_type='Bayesian',   # стабилизирует обучение
    bagging_temperature=0.8,     # усиливает рандомизацию деревьев
    grow_policy='Lossguide',     # адаптивное разрастание деревьев
    min_data_in_leaf=20          # ограничивает переобучение на малых классах
)

In [37]:
model.fit(X_train, y_train)

0:	learn: 0.2471947	total: 561ms	remaining: 9m 20s
1:	learn: 0.2992518	total: 1.04s	remaining: 8m 41s
2:	learn: 0.2987849	total: 1.5s	remaining: 8m 19s
3:	learn: 0.3639534	total: 1.97s	remaining: 8m 11s
4:	learn: 0.4084020	total: 2.43s	remaining: 8m 4s
5:	learn: 0.4625654	total: 2.93s	remaining: 8m 5s
6:	learn: 0.4752528	total: 3.43s	remaining: 8m 6s
7:	learn: 0.4762539	total: 3.87s	remaining: 8m
8:	learn: 0.4809305	total: 4.37s	remaining: 8m 1s
9:	learn: 0.5060087	total: 4.82s	remaining: 7m 57s
10:	learn: 0.5169934	total: 5.32s	remaining: 7m 58s
11:	learn: 0.5578296	total: 5.82s	remaining: 7m 58s
12:	learn: 0.5772909	total: 6.29s	remaining: 7m 57s
13:	learn: 0.5765557	total: 6.78s	remaining: 7m 57s
14:	learn: 0.5786220	total: 7.29s	remaining: 7m 58s
15:	learn: 0.5825027	total: 7.79s	remaining: 7m 58s
16:	learn: 0.6005155	total: 8.31s	remaining: 8m
17:	learn: 0.6268666	total: 8.83s	remaining: 8m 1s
18:	learn: 0.6391177	total: 9.27s	remaining: 7m 58s
19:	learn: 0.6477712	total: 9.68s	re

In [38]:
pred = model.predict(X_test)

In [39]:
print("Точность:", accuracy_score(y_test, pred))
print("\nОтчёт классификации:\n", classification_report(y_test, pred))

Точность: 0.7990562327959103

Отчёт классификации:
                                                                                   precision    recall  f1-score   support

                                                                 Благоустройство       0.89      0.75      0.81      2086
                                                                   Водоотведение       0.82      0.85      0.83        47
                                                                   Водоснабжение       0.79      0.83      0.81       162
                                                                          Кровля       0.77      0.88      0.82       165
                                  Нарушение порядка пользования общим имуществом       0.45      0.88      0.60        48
                                   Нарушение правил пользования общим имуществом       0.97      0.87      0.92       434
                         Незаконная информационная и (или) рекламная конструкция       0.81  